# DRS-HJ for Multitask Learning

Compares Douglas–Rachford splitting with analytical proximals (nuclear-norm shrinkage + Dykstra's algorithm for row/column penalties) against DRS using HJ-Prox.  Reproduces a panel of Figure 2.

## Setup


In [ ]:
# ============================================================================
# CHUNK 1: SETUP - Algorithms, Helper Functions, and Definitions
# ============================================================================

import math
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

from hj_prox import hj_prox

# Set global dtype
torch.set_default_dtype(torch.float64)
device = 'cpu'
EPS = 1e-30

# Plotting configuration
plt.rcParams.update({'font.size': 16})


# ============================================================================
# Helper Functions
# ============================================================================

def compute_penalties(B: torch.Tensor, lambda_1: float, lambda_2: float, lambda_3: float) -> torch.Tensor:
    """
    Compute the three penalties for a single matrix B.
    Returns: λ1||B||_* + λ2∑||B_i,:||_2 + λ3∑||B_:,j||_2
    """
    # Nuclear norm (sum of singular values)
    pen1 = lambda_1 * torch.linalg.svdvals(B).sum()
    
    # Row-wise ℓ2 norms
    pen2 = lambda_2 * torch.sqrt((B**2).sum(dim=1)).sum()
    
    # Column-wise ℓ2 norms  
    pen3 = lambda_3 * torch.sqrt((B**2).sum(dim=0)).sum()
    
    return pen1 + pen2 + pen3


def nuclear_penalty_batch_efficient(B_batch: torch.Tensor, p: int, q: int, 
                                  lambda_1: float) -> torch.Tensor:
    """
    Efficient nuclear norm only (when used separately in Douglas-Rachford)
    """
    if lambda_1 == 0:
        return torch.zeros(B_batch.shape[0], device=B_batch.device)
    
    B_reshaped = B_batch.reshape(-1, p, q)
    singular_values = torch.linalg.svdvals(B_reshaped)
    return lambda_1 * singular_values.sum(dim=1)


def row_col_penalty_batch_efficient(B_batch: torch.Tensor, p: int, q: int,
                                  lambda_2: float, lambda_3: float) -> torch.Tensor:
    """
    Efficient row and column norms (when used together)
    """
    B_reshaped = B_batch.reshape(-1, p, q)
    
    penalties = torch.zeros(B_batch.shape[0], device=B_batch.device)
    
    if lambda_2 > 0:
        row_norms = torch.norm(B_reshaped, p=2, dim=2)
        penalties += lambda_2 * row_norms.sum(dim=1)
    
    if lambda_3 > 0:
        col_norms = torch.norm(B_reshaped, p=2, dim=1)
        penalties += lambda_3 * col_norms.sum(dim=1)
    
    return penalties


def total_objective(B: torch.Tensor, X: torch.Tensor, Y: torch.Tensor,
                   lambda_1: float, lambda_2: float, lambda_3: float) -> torch.Tensor:
    """
    Compute total objective: 0.5||Y - XB||_F^2 + penalties
    """
    residual = Y - X @ B
    smooth_part = 0.5 * (residual**2).sum()
    penalty_part = compute_penalties(B, lambda_1, lambda_2, lambda_3)
    return smooth_part + penalty_part


def total_objective_efficient(B: torch.Tensor, X: torch.Tensor, Y: torch.Tensor,
                            lambda_1: float, lambda_2: float, lambda_3: float) -> torch.Tensor:
    """Efficient computation of total objective"""
    # Data fidelity
    residual = Y - X @ B
    smooth_part = 0.5 * (residual**2).sum()
    
    # Penalties (computed efficiently)
    if lambda_1 > 0:
        pen1 = lambda_1 * torch.linalg.svdvals(B).sum()
    else:
        pen1 = 0
        
    if lambda_2 > 0:
        pen2 = lambda_2 * torch.norm(B, p=2, dim=1).sum()
    else:
        pen2 = 0
        
    if lambda_3 > 0:
        pen3 = lambda_3 * torch.norm(B, p=2, dim=0).sum()
    else:
        pen3 = 0
    
    return smooth_part + pen1 + pen2 + pen3


# ============================================================================
# Algorithm 1: Douglas-Rachford with Analytical Proximal Operators
# ============================================================================

def douglas_rachford_analytical(
    B0: torch.Tensor,
    X: torch.Tensor,
    Y: torch.Tensor,
    lambda_1: float,
    lambda_2: float,
    lambda_3: float,
    gamma: float = 1.0,
    max_iters: int = 1000,
    prox_iters: int = 100,
    tol: float = 1e-10,
    verbose: bool = True,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Douglas-Rachford algorithm with analytical proximal operators.
    
    Splitting:
    - f(B) = 0.5||Y - XB||_F^2 + λ1||B||_* (data fidelity + nuclear norm)
    - g(B) = λ2∑||B_i,:||_2 + λ3∑||B_:,j||_2 (row and column penalties)
    
    Both proximal operators are computed analytically/iteratively.
    """
    
    xk = B0.clone()
    p, q = xk.shape
    f_hist = torch.zeros(max_iters)
    diff_hist = torch.zeros(max_iters)
    
    # Precompute for prox_f efficiency
    XtX = X.T @ X
    XtY = X.T @ Y
    n = X.shape[0]
    
    def prox_f(V: torch.Tensor, gamma: float, max_iters: int = 50) -> torch.Tensor:
        """
        Proximal operator of f(B) = 0.5||Y - XB||_F^2 + λ1||B||_*
        
        This requires solving:
        min_B { 0.5||Y - XB||_F^2 + λ1||B||_* + (1/2γ)||B - V||_F^2 }
        
        Using FISTA (Fast Iterative Shrinkage-Thresholding Algorithm)
        """
        # Lipschitz constant for the smooth part
        L_smooth = torch.linalg.eigvalsh(XtX).max() + 1.0 / gamma
        
        # Initialize
        B = V.clone()
        B_prev = B.clone()
        t = 1.0
        
        for k in range(max_iters):
            # Gradient of smooth part: X'(XB - Y) + (1/γ)(B - V)
            grad = XtX @ B - XtY + (1.0 / gamma) * (B - V)
            
            # Gradient step
            B_temp = B - (1.0 / L_smooth) * grad
            
            # Nuclear norm prox (singular value shrinkage)
            U, s, Vt = torch.linalg.svd(B_temp, full_matrices=False)
            s_thresh = torch.clamp(s - lambda_1 / L_smooth, min=0.0)
            B_new = U @ torch.diag(s_thresh) @ Vt
            
            # FISTA momentum
            t_new = (1 + math.sqrt(1 + 4 * t**2)) / 2
            B = B_new + ((t - 1) / t_new) * (B_new - B_prev)
            
            # Check convergence
            if (B_new - B_prev).norm() < 1e-8:
                break
                
            B_prev = B_new
            t = t_new
            
        return B_new
    
    def prox_g_dykstra(V: torch.Tensor, gamma: float, max_iters: int = 50) -> torch.Tensor:
        """
        Proximal operator using Dykstra's algorithm for the sum of row and column penalties.
        """
        # Initialize
        B = V.clone()
        p_rows = torch.zeros_like(V)
        p_cols = torch.zeros_like(V)
        
        for k in range(max_iters):
            B_old = B.clone()
            
            # Prox for row penalties
            Y = B + p_rows
            B_rows = torch.zeros_like(Y)
            for i in range(p):
                row_norm = Y[i, :].norm()
                if row_norm > gamma * lambda_2:
                    B_rows[i, :] = Y[i, :] * (1 - gamma * lambda_2 / row_norm)
            p_rows = Y - B_rows
            
            # Prox for column penalties
            Y = B_rows + p_cols
            B = torch.zeros_like(Y)
            for j in range(q):
                col_norm = Y[:, j].norm()
                if col_norm > gamma * lambda_3:
                    B[:, j] = Y[:, j] * (1 - gamma * lambda_3 / col_norm)
            p_cols = Y - B
            
            # Check convergence
            if (B - B_old).norm() < 1e-8:
                break
                
        return B
    
    # Main Douglas-Rachford loop
    for i in range(max_iters):
        t0 = time.time()
        
        # Step 1: y^k = prox_{γg}(x^k)
        yk = prox_g_dykstra(xk, gamma, prox_iters)
        
        # Step 2: z^k = prox_{γf}(2y^k - x^k)
        v = 2 * yk - xk
        zk = prox_f(v, gamma, prox_iters)
        
        # Step 3: x^{k+1} = x^k + (z^k - y^k)
        xk_new = xk + (zk - yk)
        
        # Compute metrics
        fk = total_objective(yk, X, Y, lambda_1, lambda_2, lambda_3)
        diff = (xk_new - xk).norm(p=2)
        primal_res = (zk - yk).norm(p=2)
        
        f_hist[i] = fk.cpu()
        diff_hist[i] = diff.item()
        xk = xk_new.clone()
        
        if verbose and i % 10 == 0:
            print(f"DRS-Analytical iter {i+1:4d}: f={fk:.6f}, ||Δx||={diff:.6f}, "
                  f"||z-y||={primal_res:.6f}, time={time.time() - t0:.4f}s")
        
        if diff < tol:
            f_hist = f_hist[:i+1]
            diff_hist = diff_hist[:i+1]
            break
    
    return yk, f_hist, diff_hist


# ============================================================================
# Algorithm 2: Douglas-Rachford with HJ-Prox
# ============================================================================

def douglas_rachford_efficient(
    B0: torch.Tensor,
    X: torch.Tensor,
    Y: torch.Tensor,
    lambda_1: float,
    lambda_2: float,
    lambda_3: float,
    gamma: float = 1.0,
    max_iters: int = 1000,
    num_samples: int = 1000,
    use_adaptive_delta: bool = True,
    tol: float = 1e-6,
    verbose: bool = True,
) -> tuple:
    """
    Douglas-Rachford with efficient penalty computations using HJ-Prox
    """
    xk = B0.clone()
    p, q = xk.shape
    device = xk.device
    
    f_hist = []
    time_hist = []
    
    def objective_f(b_batch):
        """f(B) = 0.5||Y - XB||_F^2 + λ1||B||_*"""
        B_reshaped = b_batch.reshape(-1, p, q)
        n_samples = B_reshaped.shape[0]
        
        # Vectorized data fidelity
        residuals = Y.unsqueeze(0) - torch.bmm(
            X.unsqueeze(0).expand(n_samples, -1, -1),
            B_reshaped
        )
        data_fid = 0.5 * (residuals**2).sum(dim=(1, 2))
        
        # Exact nuclear norm (efficient batch computation)
        nuclear = nuclear_penalty_batch_efficient(b_batch, p, q, lambda_1)
        
        return data_fid + nuclear
    
    def objective_g(b_batch):
        """g(B) = λ2∑||B_i,:||_2 + λ3∑||B_:,j||_2 (efficient)"""
        return row_col_penalty_batch_efficient(b_batch, p, q, lambda_2, lambda_3)
    
    if verbose:
        print(f"Starting Douglas-Rachford with HJ-Prox...")
        print(f"λ₁={lambda_1}, λ₂={lambda_2}, λ₃={lambda_3}, γ={gamma}")
        print("-" * 70)
    
    for i in range(max_iters):
        t_start = time.time()
        
        # Compute delta with annealing schedule
        delta = 200000 / (i + 1)**(2 + EPS)
        
        # Step 1: y^k = prox_{γg}(x^k)
        x_flat = xk.view(-1, 1)
        y_flat, _ = hj_prox(
            x_flat, gamma, objective_g,
            delta=delta, num_samples=num_samples,
            dtype=torch.float32, device=device,
        )
        yk = y_flat.view(p, q)
        
        # Step 2: z^k = prox_{γf}(2y^k - x^k)
        v = 2 * yk - xk
        v_flat = v.view(-1, 1)
        z_flat, _ = hj_prox(
            v_flat, gamma, objective_f,
            delta=delta, num_samples=num_samples,
            dtype=torch.float32, device=device,
        )
        zk = z_flat.view(p, q)
        
        # Step 3: x^{k+1} = x^k + (z^k - y^k)
        xk_new = xk + (zk - yk)
        
        # Compute metrics efficiently
        fk = total_objective_efficient(yk, X, Y, lambda_1, lambda_2, lambda_3)
        diff = (xk_new - xk).norm(p='fro')
        rel_diff = diff / (xk.norm(p='fro') + 1e-10)
        
        t_iter = time.time() - t_start
        f_hist.append(fk.item())
        time_hist.append(t_iter)
        
        xk = xk_new
        
        if verbose and i % 10 == 0:
            print(f"Iter {i+1:4d}: f={fk:.6f}, ||Δx||={rel_diff:.2e}, "
                  f"δ={delta:.2e}, time={t_iter:.3f}s")
        
        if rel_diff < tol:
            if verbose:
                print(f"\nConverged at iteration {i+1}")
            break
    
    return yk, torch.tensor(f_hist), torch.tensor(time_hist)



# Reproducibility seeds (added by cleanup; original notebook was unseeded).
np.random.seed(0)
torch.manual_seed(0)

print("✓ All algorithms and helper functions loaded successfully")

## Problem definition


In [ ]:
# ============================================================================
# CHUNK 2: DATA GENERATION
# ============================================================================

print("\n" + "="*60)
print("Generating multitask learning data...")
print("="*60)

# Define true coefficient patterns
beta_1a = np.array([1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0])
beta_1b = np.array([0, 0, 0, 1, 1, 0, 0, 1, 0])
beta_2a = np.array([0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0])
beta_2b = np.array([0, 1, 1, 0, 0, 0, 0, 1, 0])

# Replicate vectors
beta_1a = np.tile(beta_1a, 2)
beta_2a = np.tile(beta_2a, 2)

# True coefficient matrix
B1 = np.outer(beta_1a, beta_1b)
B2 = np.outer(beta_2a, beta_2b)
B_true = B1 + B2

# Generate data
n, sigma = 50, 1.1
X_np = np.random.normal(0, sigma, size=(n, len(beta_1a)))
E = np.random.normal(0, sigma, size=(n, len(beta_1b)))
Y_np = X_np @ B_true + E

# Convert to torch
X = torch.tensor(X_np, dtype=torch.float64)
Y = torch.tensor(Y_np, dtype=torch.float64)

# Compute Lipschitz constant
U, s, Vh = np.linalg.svd(X_np, full_matrices=False)
L = np.max(s) ** 2

print(f"✓ Data generated: X shape {X.shape}, Y shape {Y.shape}")
print(f"✓ True coefficient matrix B shape: {B_true.shape}")
print(f"✓ Lipschitz constant L = {L:.4f}")

# Initial point
B0 = torch.zeros((len(beta_1a), len(beta_1b)), dtype=torch.float64)

# Penalty parameters (same for all algorithms)
lambda_1 = 11.0  # Nuclear norm
lambda_2 = 11.0  # Row norms
lambda_3 = 11.0  # Column norms

print(f"✓ Penalty parameters: λ1={lambda_1}, λ2={lambda_2}, λ3={lambda_3}")

## Algorithm 1 — DRS with analytical proximals


In [ ]:
# ============================================================================
# CHUNK 3: RUN ALGORITHM 1 - Analytical Douglas-Rachford
# ============================================================================

print("\n" + "="*60)
print("Running Algorithm 1: Douglas-Rachford with Analytical Proximal Operators...")
print("="*60)

start_time = time.time()

B_Analytical, f_hist_Analytical, diff_hist_Analytical = douglas_rachford_analytical(
    B0=B0,
    X=X,
    Y=Y,
    lambda_1=lambda_1,
    lambda_2=lambda_2,
    lambda_3=lambda_3,
    gamma=1.0 / L * 0.005,
    max_iters=10000,
    prox_iters=100,
    tol=1e-10,
    verbose=True
)

elapsed_time = time.time() - start_time

print(f"\n✓ DRS-Analytical completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Converged in {len(f_hist_Analytical)} iterations")
print(f"  - Final objective: {f_hist_Analytical[-1]:.6f}")

## Algorithm 2 — DRS with HJ-Prox


In [ ]:
# ============================================================================
# CHUNK 4: RUN ALGORITHM 2 - Douglas-Rachford with HJ-Prox
# ============================================================================

print("\n" + "="*60)
print("Running Algorithm 2: Douglas-Rachford with HJ-Prox...")
print("="*60)

start_time = time.time()

B_HJ, f_hist_HJ, time_hist_HJ = douglas_rachford_efficient(
    B0=B0,
    X=X,
    Y=Y,
    lambda_1=lambda_1,
    lambda_2=lambda_2,
    lambda_3=lambda_3,
    gamma=1.0 / L * 0.005,
    max_iters=10000,
    num_samples=1000,
    verbose=True
)

elapsed_time = time.time() - start_time

print(f"\n✓ DRS-HJ completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Converged in {len(f_hist_HJ)} iterations")
print(f"  - Final objective: {f_hist_HJ[-1]:.6f}")

## Comparison: DRS vs DRS-HJ


In [ ]:
import os
os.makedirs('figures', exist_ok=True)
# ============================================================================
# CHUNK 5: GENERATE FIGURES
# ============================================================================

print("\n" + "="*60)
print("Generating figures...")
print("="*60)

# Compute shared vmin/vmax for consistent coloring across plots
all_vals = np.concatenate([
    B_true.flatten(),
    B_Analytical.numpy().flatten(),
    B_HJ.numpy().flatten()
])
vmin, vmax = all_vals.min(), all_vals.max()
if vmin < 0 < vmax:
    m = max(abs(vmin), abs(vmax))
    vmin, vmax = -m, m

# --- Figure 1: True Coefficient Matrix ---
plt.figure(figsize=(11, 10))
im = plt.imshow(B_true, aspect='auto', interpolation='nearest',
                cmap='RdBu_r', vmin=vmin, vmax=vmax)
plt.title('True Coefficient Matrix', fontsize=40)
plt.xlabel('Tasks (9)', fontsize=40)
plt.ylabel('Features (30)', fontsize=40)
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=40)
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig('figures/multitask_ground_truth.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 2: DRS Solution ---
plt.figure(figsize=(11, 10))
im = plt.imshow(B_Analytical.numpy(), aspect='auto', interpolation='nearest',
                cmap='RdBu_r', vmin=vmin, vmax=vmax)
plt.title('DRS', fontsize=40)
plt.xlabel('Tasks (9)', fontsize=40)
plt.ylabel('Features (30)', fontsize=40)
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=40)
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig('figures/multitask_drs.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 3: DRS-HJ Solution ---
plt.figure(figsize=(11, 10))
im = plt.imshow(B_HJ.numpy(), aspect='auto', interpolation='nearest',
                cmap='RdBu_r', vmin=vmin, vmax=vmax)
plt.title('DRS-HJ', fontsize=40)
plt.xlabel('Tasks (9)', fontsize=40)
plt.ylabel('Features (30)', fontsize=40)
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=40)
plt.gca().tick_params(axis='both', which='major', labelsize=40)
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig('figures/multitask_drs_hj.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 4: Objective Convergence ---
plt.figure(figsize=(11, 10))
plt.semilogy(f_hist_HJ.numpy(), '-', linewidth=2,
             label=f'DRS-HJ: {f_hist_HJ[-1].item():.3f}')
plt.semilogy(f_hist_Analytical.numpy(), '--', linewidth=2,
             label=f'DRS: {f_hist_Analytical[-1].item():.3f}')
plt.title('DRS Convergence', fontsize=40)
plt.xlabel('Iteration', fontsize=40)
plt.ylabel('Objective (log scale)', fontsize=40)
plt.legend(fontsize=40, loc="upper left")
plt.grid(True, alpha=0.3, which='both')
plt.gca().tick_params(axis='both', which='major', labelsize=40)
max_iter = len(f_hist_HJ)
tick_positions = np.arange(0, max_iter + 1, 2500)
plt.gca().set_xticks(tick_positions)
plt.tight_layout()
plt.savefig('figures/multitask_objectives.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All figures saved: multitask_ground_truth.pdf, multitask_drs.pdf, multitask_drs_hj.pdf, multitask_objectives.pdf")
print("\n" + "="*60)
print("✓ ALL EXPERIMENTS COMPLETED SUCCESSFULLY")
print("="*60)